In [4]:
import pandas as pd
import numpy as np
import ast

## Load Data

In [7]:
from pathlib import Path

DATA_DIR = Path('../data/raw')

train = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8', encoding_errors='replace')
val   = pd.read_csv(DATA_DIR / 'val.csv',   encoding='utf-8', encoding_errors='replace')
test  = pd.read_csv(DATA_DIR / 'test.csv',  encoding='utf-8', encoding_errors='replace')


print(train.shape, val.shape, test.shape)

(16531, 3) (2066, 3) (2067, 3)


In [9]:
train.head(10)

,id,text,labels
0,tik000008,Xem mà ngẫm lại cuộc đời bản thân ta đã trải q...,[12]
1,5743,bức ảnh xuất sắc ❤️,"[2, 8, 3]"
2,32895,"Vừa đẹp trai, vừa tài giỏi. Nhà mặt phố, bố là...","[8, 7]"
3,you001182,"Bài học: <br>1: 5 nhìn, 4 chạm, 3 nghe, 2 ngửi...",[27]
4,12052,Dima Egiazarov bởi vì chúng tôi là người Việt ...,"[24, 23]"
5,1824,giờ mới biết,"[12, 13]"
6,tik023326,cảm thấy tự hào về đất nước mik,[7]
7,1046,nhìn mặt là cười phọt rồi,[0]
8,11556,Cris Minh Algeria nó theo đạo hồi. Ko đc chuyể...,[27]
9,2320,nó giống tao ghê . gặp tao tao cũng chọn mày,"[0, 11, 2, 3]"


## Parse cột labels thành list

In [10]:
train['labels'] = train['labels'].apply(ast.literal_eval)
val['labels']   = val['labels'].apply(ast.literal_eval)
test['labels']  = test['labels'].apply(ast.literal_eval)

print(type(train['labels'][0]))  # kiểm tra kiểu dữ liệu

<class 'list'>


## Label distribution

In [ ]:
from collections import Counter

label_counts = Counter(label for labels in train['labels'] for label in labels)
print(label_counts.most_common(10))

[(0, 2868), (21, 2785), (25, 2662), (2, 1614), (24, 1589), (20, 1522), (23, 1206), (3, 1175), (6, 1062), (5, 1046)]


amusement (2,868) chiếm gần gấp đôi các nhãn phổ biến khác. Phân bố không đồng đều rõ rệt ngay ở top 10.

### Nhãn ít nhất:

In [12]:
print(label_counts.most_common()[-10:])

[(9, 765), (4, 762), (22, 747), (16, 745), (17, 728), (18, 707), (12, 687), (15, 676), (13, 651), (10, 635)]


relief (635) thấp hơn amusement 4.5 lần. Các nhãn thiểu số như disapproval, surprise, relief sẽ khó học với threshold 0.5 cố định, đây là bằng chứng cho RQ3.

### Trung bình số nhãn mỗi câu:

In [14]:
train['num_labels'] = train['labels'].apply(len)
print(train['num_labels'].describe())

count    16531.000000
mean         1.908233
std          0.790437
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          5.000000
Name: num_labels, dtype: float64


Median: 2 nhãn/câu, max: 5 nhãn. Dataset thực sự multi-label thay vì single-label. Model phải học dự đoán nhiều cảm xúc đồng thời.

### Text analysis

In [16]:
train['text_length'] = train['text'].apply(len)
print(train['text_length'].describe())

count    16531.000000
mean        66.509407
std         66.533940
min          1.000000
25%         28.000000
50%         47.000000
75%         81.000000
max        906.000000
Name: text_length, dtype: float64


75% câu dưới 81 ký tự. Có thể dùng max_length=128 cho BERT thay vì 512 mặc định giúp tiết kiệm bộ nhớ và tăng tốc training đáng kể.

### Emoji analysis

In [18]:
import emoji

train['has_emoji'] = train['text'].apply(lambda x: bool(emoji.emoji_list(x)))
train['emoji_count'] = train['text'].apply(lambda x: len(emoji.emoji_list(x)))

train['has_emoji'].value_counts()

has_emoji
False    12422
True      4109
Name: count, dtype: int64

In [19]:
train.loc[train['has_emoji'], 'emoji_count'].describe()

count    4109.00000
mean        1.90825
std         2.18800
min         1.00000
25%         1.00000
50%         1.00000
75%         2.00000
max        48.00000
Name: emoji_count, dtype: float64

24.9% câu chứa emoji, chiếm gần 1/4 tập train. Trung bình 1.91 emoji/câu (trong số câu có emoji). Đủ lớn để chiến lược xử lý emoji ảnh hưởng đến kết quả model, bằng chứng trực tiếp cho RQ2.

### Teencode frequency

In [ ]:
TEENCODE = ['k', 'ko', 'kg', 'kh', 'đc', 'dc', 'vs', 'mn', 'mk', 
            'mik', 'bn', 'bh', 'nx', 'cx', 'ck', 'vk', 'bt', 'tl',
            'rep', 'cmt', 'like', 'share', 'fb', 'ib', 'dm', 'vloz']

import re

def has_teencode(text):
    words = re.findall(r'\b\w+\b', text.lower())
    return any(w in TEENCODE for w in words)

train['has_teencode'] = train['text'].apply(has_teencode)

train['has_teencode'].value_counts()

has_teencode
False    13999
True      2532
Name: count, dtype: int64

In [ ]:
train['has_teencode'].value_counts(normalize=True)

has_teencode
False    0.846833
True     0.153167
Name: proportion, dtype: float64

15.3% câu chứa teencode. Ít hơn emoji (24.9%) về tần suất, nhưng tác động lớn hơn về chất lượng vì teencode khiến BERT tokenize sai ngữ nghĩa hoàn toàn. Bằng chứng cho RQ1.